In [ ]:
!pip install -q groq gradio

print("✅ EVA libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.0 MB/s eta 0:00:00
✅ EVA libraries installed successfully!


In [ ]:
# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================

# Groq Python SDK
from groq import Groq

# Secure access to Google Colab Secrets
from google.colab import userdata

# Gradio for our chatbot interface
import gradio as gr

# JSON for structured emotion analysis
import json

# Time is used for handling temporary API errors
import time

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [ ]:
# ============================================================
# CELL 3: CONFIGURE GROQ API
# ============================================================

# Read the API key from Colab Secrets.
# The actual key will NOT be printed.

GROQ_API_KEY = userdata.get('aaditi21')

# Create the Groq client.
groq_client = Groq(
    api_key=GROQ_API_KEY
)

print("✅ Groq API configured successfully!")

✅ Groq API configured successfully!


In [ ]:
# ============================================================
# CELL 4: TEST GROQ
# ============================================================

response = groq_client.chat.completions.create(

    # Generative AI model
    model="openai/gpt-oss-20b",

    # System instruction + user message
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        },
        {
            "role": "user",
            "content": "Explain Generative AI in two simple sentences."
        }
    ],

    # Limit the response length
    max_completion_tokens=150
)

# Display the generated answer
print("🤖 GROQ RESPONSE")
print("=" * 50)
print(response.choices[0].message.content)

🤖 GROQ RESPONSE
Generative AI creates new content—like text, images, or music—by learning patterns from large amounts of data. It uses those patterns to produce original outputs that resemble the input material but aren’t copied from it.


In [ ]:
# ============================================================
# CELL 5: EVA EMOTION UNDERSTANDING
# ============================================================

def understand_emotion(user_message):

    # --------------------------------------------------------
    # Prompt the Generative AI model
    # --------------------------------------------------------

    prompt = f"""
You are the emotion analysis module of
EmotionallyVersatileAI (EVA).

Analyze the user's message carefully.

USER MESSAGE:
{user_message}

Identify the following:

1. emotion:
   Choose the most appropriate primary emotion.
   Examples: Joy, Sadness, Anger, Fear, Anxiety,
   Surprise, Disgust, Frustration, Excitement, Neutral.

2. intensity:
   Choose only Low, Medium, or High.

3. intent:
   Identify what the user wants.
   Examples: Seeking Help, Seeking Advice,
   Asking Information, Complaint, Sharing Feelings,
   Casual Conversation, etc.

4. context:
   Briefly explain the situation.

5. response_style:
   Explain how the AI should communicate with this user.

Return ONLY valid JSON.

Use exactly this format:

{{
    "emotion": "Fear",
    "intensity": "High",
    "intent": "Seeking Help",
    "context": "User is worried about an upcoming presentation",
    "response_style": "Reassuring, supportive and practical"
}}
"""

    # --------------------------------------------------------
    # Send request to Groq
    # --------------------------------------------------------

    response = groq_client.chat.completions.create(

        # LLM used by EVA
        model="openai/gpt-oss-20b",

        # System role defines the AI's job
        messages=[
            {
                "role": "system",
                "content":
                "You are EVA's emotion analysis engine. "
                "Return only valid JSON."
            },

            # User message + analysis instructions
            {
                "role": "user",
                "content": prompt
            }
        ],

        # Lower temperature makes structured output
        # more consistent.
        temperature=0.2,

        # Limit generated output
        max_completion_tokens=300
    )

    # --------------------------------------------------------
    # Extract the generated text
    # --------------------------------------------------------

    result = response.choices[0].message.content.strip()

    # Remove markdown formatting if the model adds it
    if result.startswith("```"):
        result = result.replace("```json", "")
        result = result.replace("```", "")
        result = result.strip()

    # Convert JSON text into a Python dictionary
    emotion_data = json.loads(result)

    return emotion_data


print("✅ EVA emotion understanding engine created!")

✅ EVA emotion understanding engine created!


In [ ]:
# ============================================================
# CELL 6: TEST EMOTION UNDERSTANDING
# ============================================================

test_message = """
I have my presentation tomorrow and I am really scared.
I studied a lot but I feel like I will forget everything.
Can you help me?
"""

# Analyze the user's message
emotion_result = understand_emotion(test_message)

# Display the analysis
print("🧠 EVA EMOTION ANALYSIS")
print("=" * 60)

print("Emotion       :", emotion_result["emotion"])
print("Intensity     :", emotion_result["intensity"])
print("Intent        :", emotion_result["intent"])
print("Context       :", emotion_result["context"])
print("Response Style:", emotion_result["response_style"])

🧠 EVA EMOTION ANALYSIS
Emotion       : Fear
Intensity     : High
Intent        : Seeking Help
Context       : User is worried about an upcoming presentation
Response Style: Reassuring, supportive and practical


In [ ]:
# ============================================================
# CELL 7: EVA ADAPTIVE RESPONSE GENERATOR
# ============================================================

def generate_adaptive_response(user_message, emotion_data):

    # --------------------------------------------------------
    # Extract the information identified by EVA
    # --------------------------------------------------------

    emotion = emotion_data["emotion"]
    intensity = emotion_data["intensity"]
    intent = emotion_data["intent"]
    context = emotion_data["context"]
    response_style = emotion_data["response_style"]

    # --------------------------------------------------------
    # Create an adaptive response prompt
    # --------------------------------------------------------

    prompt = f"""
You are EVA (EmotionallyVersatileAI),
an emotion-aware Generative AI assistant.

USER MESSAGE:
{user_message}

EMOTIONAL ANALYSIS:

Emotion:
{emotion}

Intensity:
{intensity}

Intent:
{intent}

Context:
{context}

Recommended Response Style:
{response_style}

Your task is to generate the best possible response.

IMPORTANT RULES:

1. Address the user's actual question or need.
2. Adapt your tone according to the detected emotion.
3. Respect the emotional intensity.
4. Do not mention that you detected or analyzed the emotion.
5. Do not sound robotic.
6. Be natural and conversational.
7. If the user is afraid, be reassuring and practical.
8. If the user is angry, remain calm and non-defensive.
9. If the user is sad, be empathetic and supportive.
10. If the user is happy or excited, match their positive energy.
11. If the user asks a technical question, still provide
    the technical information while adapting the tone.
12. Do not claim to be a human or mental-health professional.

Generate only the final response to the user.
"""

    # --------------------------------------------------------
    # Send the request to Groq
    # --------------------------------------------------------

    response = groq_client.chat.completions.create(

        model="openai/gpt-oss-20b",

        messages=[
            {
                "role": "system",
                "content":
                "You are EVA, an emotionally adaptive "
                "Generative AI assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        # Allows natural and varied responses
        temperature=0.7,

        # Keep response reasonably short
        max_completion_tokens=300
    )

    # --------------------------------------------------------
    # Return the generated response
    # --------------------------------------------------------

    return response.choices[0].message.content.strip()


print("✅ EVA adaptive response generator created!")

✅ EVA adaptive response generator created!


In [ ]:
# ============================================================
# CELL 8: TEST ADAPTIVE RESPONSE
# ============================================================

# Generate a response using the emotional information
adaptive_response = generate_adaptive_response(
    test_message,
    emotion_result
)

print("🤖 EVA ADAPTIVE RESPONSE")
print("=" * 60)
print(adaptive_response)

🤖 EVA ADAPTIVE RESPONSE
Hey, I totally get how nerve‑wracking that can feel. A lot of people think they’ll blank out, but the truth is you’ve already done the heavy lifting by studying hard. Here’s a quick game plan to keep those key points fresh and calm your nerves:

| Step | What to do | Why it helps |
|------|------------|--------------|
| **1. Pick 3–5 core ideas** | Write them on a single sheet or a sticky note. | When you’re in the moment, a few anchors are easier to recall than a long list. |
| **2. Teach it to someone** | Pretend you’re explaining it to a friend, a pet, or even a mirror. | The act of verbalizing forces the material into memory and highlights gaps. |
| **3. Do a “quick run‑through”** | Go through your outline aloud for 2–3 minutes—no notes, just your own words. | This mimics the real flow and trains your brain to retrieve the information on cue. |
| **4. Visualize success** | Close your eyes and picture yourself standing confidently, the audience engaged, and


In [ ]:
# ============================================================
# CELL 9: COMPLETE EVA PIPELINE
# ============================================================

def eva_process(user_message):

    # --------------------------------------------------------
    # STEP 1: Analyze the user's emotional state
    # --------------------------------------------------------

    print("🧠 Analyzing emotional state...")

    emotion_data = understand_emotion(user_message)

    print("✅ Emotion analysis completed!")

    # --------------------------------------------------------
    # STEP 2: Generate an adaptive response
    # --------------------------------------------------------

    print("🤖 Generating adaptive response...")

    adaptive_response = generate_adaptive_response(
        user_message,
        emotion_data
    )

    print("✅ Adaptive response generated!")

    # --------------------------------------------------------
    # STEP 3: Return complete result
    # --------------------------------------------------------

    return {
        "emotion_data": emotion_data,
        "response": adaptive_response
    }


print("✅ Complete EVA pipeline created!")

✅ Complete EVA pipeline created!


In [ ]:
# ============================================================
# CELL 10: TEST COMPLETE EVA
# ============================================================

user_message = """
I am really nervous about my presentation tomorrow.
I studied a lot but I am afraid I will forget everything.
Can you help me?
"""

# Run the complete EVA pipeline
eva_result = eva_process(user_message)

# ------------------------------------------------------------
# Display emotion analysis
# ------------------------------------------------------------

emotion = eva_result["emotion_data"]

print("\n" + "=" * 65)
print("🧠 EVA EMOTION ANALYSIS")
print("=" * 65)

print("Emotion       :", emotion["emotion"])
print("Intensity     :", emotion["intensity"])
print("Intent        :", emotion["intent"])
print("Context       :", emotion["context"])
print("Response Style:", emotion["response_style"])

# ------------------------------------------------------------
# Display adaptive response
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("🤖 EVA ADAPTIVE RESPONSE")
print("=" * 65)

print(eva_result["response"])

🧠 Analyzing emotional state...
✅ Emotion analysis completed!
🤖 Generating adaptive response...
✅ Adaptive response generated!

🧠 EVA EMOTION ANALYSIS
Emotion       : Anxiety
Intensity     : High
Intent        : Seeking Help
Context       : User is worried about an upcoming presentation and fears forgetting what they studied
Response Style: Reassuring, supportive and practical

🤖 EVA ADAPTIVE RESPONSE
Hey, I totally get how nerves can sneak up on you before a big talk. You’ve already put in the work, so you’ve got a solid foundation to lean on. Here are a few quick things that might help you feel more confident and keep the material fresh:

1. **Create a “cheat‑sheet” of key points**  
   Write one sentence for each slide or main idea. Keep it in a small notebook or a sticky note on your phone. When you’re rehearsing, read it aloud—this trains your brain to link the detail to the headline.

2. **Practice in the “real” setting**  
   If you can, set up a mock room or stand in front of a 

In [ ]:
# ============================================================
# CELL 11: MULTIPLE EMOTIONAL SCENARIOS
# ============================================================

test_messages = [

    # Scenario 1: Fear
    "I am really scared about my exam tomorrow. "
    "I don't think I will remember anything.",

    # Scenario 2: Anger
    "This website is terrible! "
    "I have been trying to login for an hour!",

    # Scenario 3: Sadness
    "I studied so hard but I failed my exam. "
    "I feel completely hopeless.",

    # Scenario 4: Joy
    "I finally completed my project and got an A grade! "
    "I am so happy!",

    # Scenario 5: Neutral
    "Can you explain what a database management system is?"
]


# ------------------------------------------------------------
# Process each scenario
# ------------------------------------------------------------

for i, message in enumerate(test_messages, start=1):

    print("\n" + "=" * 75)
    print(f"TEST CASE {i}")
    print("=" * 75)

    print("\n👤 USER:")
    print(message)

    # Run EVA
    result = eva_process(message)

    emotion = result["emotion_data"]

    print("\n🧠 EMOTION:")
    print(emotion["emotion"])

    print("🔥 INTENSITY:")
    print(emotion["intensity"])

    print("🎯 INTENT:")
    print(emotion["intent"])

    print("\n🤖 EVA RESPONSE:")
    print(result["response"])


TEST CASE 1

👤 USER:
I am really scared about my exam tomorrow. I don't think I will remember anything.
🧠 Analyzing emotional state...
✅ Emotion analysis completed!
🤖 Generating adaptive response...
✅ Adaptive response generated!

🧠 EMOTION:
Fear
🔥 INTENSITY:
High
🎯 INTENT:
Seeking Help

🤖 EVA RESPONSE:
Hey, I totally get how stressful that can feel. Let’s try to turn the panic into a quick, focused prep session that leaves you feeling a bit more confident.

**1. Do a rapid “snapshot” review.**  
- Grab a notebook or a sheet of

TEST CASE 2

👤 USER:
This website is terrible! I have been trying to login for an hour!
🧠 Analyzing emotional state...
✅ Emotion analysis completed!
🤖 Generating adaptive response...
✅ Adaptive response generated!

🧠 EMOTION:
Frustration
🔥 INTENSITY:
High
🎯 INTENT:
Seeking Help

🤖 EVA RESPONSE:
I’m really sorry you’re stuck—this shouldn’t be happening. Let’s try a few quick fixes to get you back in:

1. **Double‑check your credentials** – make sure the caps lo

In [ ]:
# ============================================================
# FIXED EVA PROCESS
# ============================================================

def eva_process(user_message):

    prompt = f"""
You are EVA (EmotionallyVersatileAI), an emotion-aware
Generative AI assistant.

Understand the meaning and context of the user's message.
Do not depend only on emotion keywords.

USER MESSAGE:
{user_message}

Determine:

1. Primary emotion
2. Intensity: Low, Medium, or High
3. User intent
4. Context
5. Response style

Then generate a response that naturally matches
the user's emotional state.

Possible emotions include:
Fear, Anxiety, Anger, Frustration, Sadness,
Joy, Excitement, Surprise, Disgust, Neutral.

Return ONLY valid JSON.
Do not use markdown.
Do not write anything before or after the JSON.

Format:

{{
    "emotion": "Fear",
    "intensity": "High",
    "intent": "Seeking Help",
    "context": "User is worried about an upcoming presentation",
    "response_style": "Reassuring and practical",
    "response": "Your helpful response here"
}}
"""

    try:

        print("🧠 EVA is analyzing the message...")

        response = groq_client.chat.completions.create(

            model="openai/gpt-oss-20b",

            messages=[
                {
                    "role": "system",
                    "content":
                    "You are EVA. Return ONLY valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.2,

            max_completion_tokens=500
        )

        # Get raw LLM output
        raw_result = response.choices[0].message.content.strip()

        print("✅ Groq response received!")

        # ----------------------------------------------------
        # Clean possible markdown formatting
        # ----------------------------------------------------

        if "```json" in raw_result:
            raw_result = raw_result.replace("```json", "")

        if "```" in raw_result:
            raw_result = raw_result.replace("```", "")

        raw_result = raw_result.strip()

        # ----------------------------------------------------
        # Convert JSON response into Python dictionary
        # ----------------------------------------------------

        result = json.loads(raw_result)

        print("✅ EVA analysis completed!")

        return {
            "emotion_data": {
                "emotion": result["emotion"],
                "intensity": result["intensity"],
                "intent": result["intent"],
                "context": result["context"],
                "response_style": result["response_style"]
            },

            "response": result["response"]
        }

    except Exception as error:

        # IMPORTANT:
        # Show the real error instead of hiding it.
        print("❌ EVA ERROR:")
        print(type(error).__name__)
        print(error)

        return {
            "emotion_data": {
                "emotion": "Unavailable",
                "intensity": "Unknown",
                "intent": "Unknown",
                "context": "Processing error",
                "response_style": "General"
            },

            "response":
            "EVA could not process this message. "
            "Please try again."
        }


print("✅ EVA process updated!")

✅ EVA process updated!


In [ ]:
# ============================================================
# TEST EVA
# ============================================================

test_message = """
My presentation is tomorrow. I prepared everything,
but I keep imagining myself forgetting all the answers
in front of everyone.
"""

result = eva_process(test_message)

print("\n" + "=" * 60)
print("EMOTION:", result["emotion_data"]["emotion"])
print("INTENSITY:", result["emotion_data"]["intensity"])
print("INTENT:", result["emotion_data"]["intent"])

print("\n🤖 EVA:")
print(result["response"])

🧠 EVA is analyzing the message...
✅ Groq response received!
✅ EVA analysis completed!

EMOTION: Fear
INTENSITY: High
INTENT: Seeking Help

🤖 EVA:
It’s completely normal to feel nervous before a big presentation—your brain is just trying to protect you. Here are a few quick steps that can help you feel more prepared and calm:
1. **Practice the key points aloud**: Even if you’re not rehearsing the whole talk, run through the main ideas and any tough questions you anticipate. This builds muscle memory.
2. **Use a cue card**: Write one sentence per slide or topic on a small card. When you feel stuck, glance at it—just enough to jog your memory without looking like you’re reading.
3. **Visualize success**: Spend 2–3 minutes picturing yourself speaking confidently, the audience nodding, and the presentation ending smoothly. This mental rehearsal can reduce anxiety.
4. **Breathe and pause**: If you feel a wave of panic, take a slow breath, pause for a beat, and then continue. A brief pause ca

In [ ]:
# ============================================================
# CELL 14: SEMANTIC EMOTION TESTING
# ============================================================

test_cases = [

    # Fear without directly saying "fear"
    """
    Tomorrow is my presentation and I keep thinking
    that I might mess everything up.
    """,

    # Anger without directly saying "anger"
    """
    I have tried logging into this website for almost
    an hour. Nothing works and I am completely fed up.
    """,

    # Sadness without directly saying "sad"
    """
    I worked on this project for weeks and after
    seeing the result, I just feel empty.
    """,

    # Joy without directly saying "happy"
    """
    After months of work, everything finally came together.
    We got the result we were hoping for!
    """,

    # Neutral
    """
    What is the difference between SQL and NoSQL databases?
    """
]


# ------------------------------------------------------------
# Run each test case
# ------------------------------------------------------------

for i, message in enumerate(test_cases, start=1):

    print("\n" + "=" * 70)
    print(f"TEST CASE {i}")
    print("=" * 70)

    print("USER:")
    print(message.strip())

    # Run EVA
    result = eva_process(message)

    emotion = result["emotion_data"]

    print("\n🧠 EMOTION:", emotion["emotion"])
    print("🔥 INTENSITY:", emotion["intensity"])
    print("🎯 INTENT:", emotion["intent"])

    print("\n🤖 EVA:")
    print(result["response"])


TEST CASE 1
USER:
Tomorrow is my presentation and I keep thinking
    that I might mess everything up.
🧠 EVA is understanding the message...
❌ EVA API error:
Unterminated string starting at: line 7 column 15 (char 250)

🧠 EMOTION: Unavailable
🔥 INTENSITY: Unknown
🎯 INTENT: Unknown

🤖 EVA:
EVA is temporarily unavailable. Please try again.

TEST CASE 2
USER:
I have tried logging into this website for almost
    an hour. Nothing works and I am completely fed up.
🧠 EVA is understanding the message...
❌ EVA API error:
Unterminated string starting at: line 7 column 17 (char 249)

🧠 EMOTION: Unavailable
🔥 INTENSITY: Unknown
🎯 INTENT: Unknown

🤖 EVA:
EVA is temporarily unavailable. Please try again.

TEST CASE 3
USER:
I worked on this project for weeks and after
    seeing the result, I just feel empty.
🧠 EVA is understanding the message...
❌ EVA API error:
Unterminated string starting at: line 7 column 15 (char 258)

🧠 EMOTION: Unavailable
🔥 INTENSITY: Unknown
🎯 INTENT: Unknown

🤖 EVA:
EVA i

In [ ]:
# ============================================================
# CELL 15: EVA CONVERSATION MEMORY
# ============================================================

# Stores the emotional history of the current conversation
emotion_history = []


def store_emotion(result):

    # Get the emotion analysis
    emotion = result["emotion_data"]

    # Store important emotional information
    emotion_history.append({
        "emotion": emotion["emotion"],
        "intensity": emotion["intensity"],
        "intent": emotion["intent"]
    })

    return emotion_history


print("✅ EVA conversation memory created!")

✅ EVA conversation memory created!


In [ ]:
# ============================================================
# CELL 16: EMOTION EVOLUTION
# ============================================================

# Start a fresh emotional history
emotion_history = []

conversation = [

    "I have my presentation tomorrow and I don't know "
    "if I am prepared enough.",

    "I practiced the presentation twice and it actually "
    "went better than I expected.",

    "My presentation is finally over and my teacher "
    "said it was really good!"
]


# ------------------------------------------------------------
# Process the conversation
# ------------------------------------------------------------

for message in conversation:

    print("\n" + "-" * 65)
    print("USER:", message)

    # Analyze message
    result = eva_process(message)

    # Store emotion
    history = store_emotion(result)

    print("EMOTION:", result["emotion_data"]["emotion"])
    print("INTENSITY:", result["emotion_data"]["intensity"])

    print("\nEVA:")
    print(result["response"])


# ------------------------------------------------------------
# Display emotional journey
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("📈 EVA EMOTIONAL JOURNEY")
print("=" * 65)

for i, item in enumerate(emotion_history, start=1):

    print(
        f"Step {i}: "
        f"{item['emotion']} "
        f"({item['intensity']})"
    )


-----------------------------------------------------------------
USER: I have my presentation tomorrow and I don't know if I am prepared enough.
🧠 EVA is understanding the message...
❌ EVA API error:
Unterminated string starting at: line 7 column 17 (char 258)
EMOTION: Unavailable
INTENSITY: Unknown

EVA:
EVA is temporarily unavailable. Please try again.

-----------------------------------------------------------------
USER: I practiced the presentation twice and it actually went better than I expected.
🧠 EVA is understanding the message...
❌ EVA API error:
Unterminated string starting at: line 7 column 17 (char 239)
EMOTION: Unavailable
INTENSITY: Unknown

EVA:
EVA is temporarily unavailable. Please try again.

-----------------------------------------------------------------
USER: My presentation is finally over and my teacher said it was really good!
🧠 EVA is understanding the message...
✅ Semantic analysis completed!
EMOTION: Joy
INTENSITY: High

EVA:
Congratulations! I'm thrill

In [ ]:
# ============================================================
# CELL 17: EMOTION HISTORY FOR GRADIO
# ============================================================

# Stores emotions during the current EVA session
emotion_history = []


def eva_chat_with_history(user_message):

    # --------------------------------------------------------
    # Check empty input
    # --------------------------------------------------------

    if not user_message or not user_message.strip():

        return (
            "—",
            "—",
            "—",
            "—",
            "—",
            "Please enter a message.",
            "No emotional journey yet."
        )

    # --------------------------------------------------------
    # Run EVA
    # --------------------------------------------------------

    result = eva_process(user_message)

    emotion = result["emotion_data"]

    # --------------------------------------------------------
    # Store current emotional state
    # --------------------------------------------------------

    emotion_history.append(
        {
            "emotion": emotion["emotion"],
            "intensity": emotion["intensity"]
        }
    )

    # --------------------------------------------------------
    # Create readable emotional journey
    # --------------------------------------------------------

    journey = " → ".join(
        [
            f"{item['emotion']} ({item['intensity']})"
            for item in emotion_history
        ]
    )

    # --------------------------------------------------------
    # Return results to Gradio
    # --------------------------------------------------------

    return (
        emotion["emotion"],
        emotion["intensity"],
        emotion["intent"],
        emotion["context"],
        emotion["response_style"],
        result["response"],
        journey
    )


print("✅ Emotion history system created!")

✅ Emotion history system created!


In [ ]:
# ============================================================
# FINAL EVA CONVERSATIONAL CHAT
# ============================================================

import gradio as gr


def eva_conversation(user_message, history):

    # Check empty message
    if not user_message.strip():
        return history, ""

    # --------------------------------------------------------
    # Convert Gradio chat history into simple text
    # --------------------------------------------------------

    conversation = ""

    for message in history:

        # Gradio message format
        if isinstance(message, dict):

            role = message.get("role")
            content = message.get("content")

            conversation += f"{role.upper()}: {content}\n"

    # --------------------------------------------------------
    # Send current message + previous conversation to EVA
    # --------------------------------------------------------

    full_prompt = f"""
PREVIOUS CONVERSATION:
{conversation}

CURRENT USER MESSAGE:
{user_message}

Use the previous conversation when relevant.
Understand the user's current emotion and context,
then generate an appropriate response.
"""

    # Use our existing EVA engine
    result = eva_process(full_prompt)

    emotion = result["emotion_data"]

    response = (
        result["response"]
        + "\n\n🧠 Emotion: "
        + emotion["emotion"]
        + " | Intensity: "
        + emotion["intensity"]
    )

    # --------------------------------------------------------
    # Add messages to chat history
    # --------------------------------------------------------

    history.append({
        "role": "user",
        "content": user_message
    })

    history.append({
        "role": "assistant",
        "content": response
    })

    return history, ""


# ============================================================
# CREATE CHAT INTERFACE
# ============================================================

with gr.Blocks(
    title="EmotionallyVersatileAI - EVA"
) as eva_app:

    gr.Markdown(
        """
        # 🧠 EmotionallyVersatileAI (EVA)

        ### *AI that understands the person, not just the query.*

        **GenAI-powered Emotion-Aware Conversational Assistant**
        """
    )

    gr.Markdown("---")

    # Chat window
    chatbot = gr.Chatbot(
        label="💬 Chat with EVA",
        height=500
    )

    # Message box
    message_box = gr.Textbox(
        label="Your Message",
        placeholder="Continue your conversation with EVA...",
        lines=2
    )

    with gr.Row():

        send_button = gr.Button("🚀 Send")
        clear_button = gr.Button("🗑️ Clear")

    # Send button
    send_button.click(
        fn=eva_conversation,
        inputs=[message_box, chatbot],
        outputs=[chatbot, message_box]
    )

    # Enter key
    message_box.submit(
        fn=eva_conversation,
        inputs=[message_box, chatbot],
        outputs=[chatbot, message_box]
    )

    # Clear chat
    clear_button.click(
        fn=lambda: [],
        inputs=None,
        outputs=chatbot
    )


# Launch
eva_app.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f59a58f6b7f8b6ebe7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# DEBUG: CHECK GROQ CONNECTION
# ============================================================

try:

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",

        messages=[
            {
                "role": "user",
                "content": "Say hello in one sentence."
            }
        ],

        max_completion_tokens=50
    )

    print("✅ GROQ IS WORKING!")
    print("-" * 50)
    print(response.choices[0].message.content)

except Exception as error:

    print("❌ GROQ ERROR")
    print("-" * 50)
    print(type(error).__name__)
    print(error)

✅ GROQ IS WORKING!
--------------------------------------------------



In [ ]:
# ============================================================
# FINAL EVA UI
# CONTINUOUS CHAT + EMOTION ANALYSIS
# ============================================================

import gradio as gr


# ------------------------------------------------------------
# EVA CHAT FUNCTION
# ------------------------------------------------------------

def eva_chat(user_message, history):

    # Check empty message
    if not user_message.strip():
        return (
            history,
            "",
            "—",
            "—",
            "—",
            "—",
            "—",
            ""
        )

    # --------------------------------------------------------
    # Build previous conversation
    # --------------------------------------------------------

    conversation = ""

    for message in history:

        if isinstance(message, dict):

            role = message.get("role", "")
            content = message.get("content", "")

            conversation += (
                f"{role}: {content}\n"
            )

    # --------------------------------------------------------
    # Combine conversation + current message
    # --------------------------------------------------------

    if conversation:

        full_message = f"""
Previous conversation:

{conversation}

Current user message:

{user_message}

Continue the conversation naturally.
Use the previous conversation when relevant.
Answer the current message appropriately.
"""

    else:

        full_message = user_message

    # --------------------------------------------------------
    # Send to EVA
    # --------------------------------------------------------

    result = eva_process(full_message)

    emotion = result["emotion_data"]
    response = result["response"]

    # --------------------------------------------------------
    # Add USER message as dictionary
    # --------------------------------------------------------

    history.append({
        "role": "user",
        "content": user_message
    })

    # --------------------------------------------------------
    # Add EVA message as dictionary
    # --------------------------------------------------------

    history.append({
        "role": "assistant",
        "content": response
    })

    # --------------------------------------------------------
    # Return everything
    # --------------------------------------------------------

    return (
        history,
        "",

        emotion["emotion"],
        emotion["intensity"],
        emotion["intent"],
        emotion["context"],
        emotion["response_style"],
        response
    )


# ============================================================
# CLEAR FUNCTION
# ============================================================

def clear_eva():

    return (
        [],
        "",
        "—",
        "—",
        "—",
        "—",
        "—",
        ""
    )


# ============================================================
# CREATE APPLICATION
# ============================================================

with gr.Blocks(
    title="EmotionallyVersatileAI - EVA"
) as eva_app:

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    gr.Markdown(
        """
        # 🧠 EmotionallyVersatileAI (EVA)

        ### *AI that understands the person, not just the query.*

        **GenAI-powered Emotion-Aware Conversational Assistant**
        """
    )

    gr.Markdown("---")

    # ========================================================
    # CHAT
    # ========================================================

    gr.Markdown("## 💬 Chat with EVA")

    chatbot = gr.Chatbot(
        label="EVA Conversation",
        height=350
    )

    user_input = gr.Textbox(
        label="Your Message",
        placeholder="Continue your conversation with EVA...",
        lines=3
    )

    with gr.Row():

        send_button = gr.Button(
            "🚀 Send",
            variant="primary"
        )

        clear_button = gr.Button(
            "🗑️ Clear"
        )

    # ========================================================
    # EMOTION ANALYSIS
    # ========================================================

    gr.Markdown("## 🧠 EVA Emotion Analysis")

    with gr.Row():

        emotion_output = gr.Textbox(
            label="Emotion"
        )

        intensity_output = gr.Textbox(
            label="Intensity"
        )

        intent_output = gr.Textbox(
            label="Intent"
        )

    context_output = gr.Textbox(
        label="📝 Context"
    )

    strategy_output = gr.Textbox(
        label="💬 Response Strategy"
    )

    # ========================================================
    # ADAPTIVE RESPONSE
    # ========================================================

    gr.Markdown("## 🤖 EVA Adaptive Response")

    response_output = gr.Textbox(
        label="EVA Response",
        lines=6
    )

    # ========================================================
    # SEND BUTTON
    # ========================================================

    send_button.click(
        fn=eva_chat,

        inputs=[
            user_input,
            chatbot
        ],

        outputs=[
            chatbot,
            user_input,
            emotion_output,
            intensity_output,
            intent_output,
            context_output,
            strategy_output,
            response_output
        ]
    )

    # ========================================================
    # ENTER KEY
    # ========================================================

    user_input.submit(
        fn=eva_chat,

        inputs=[
            user_input,
            chatbot
        ],

        outputs=[
            chatbot,
            user_input,
            emotion_output,
            intensity_output,
            intent_output,
            context_output,
            strategy_output,
            response_output
        ]
    )

    # ========================================================
    # CLEAR CHAT
    # ========================================================

    clear_button.click(
        fn=clear_eva,
        inputs=None,
        outputs=[
            chatbot,
            user_input,
            emotion_output,
            intensity_output,
            intent_output,
            context_output,
            strategy_output,
            response_output
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

eva_app.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://61964439306d1d0793.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
